# Token-level probing with POS

POS (Parts Of Speech) is a key element of linguistic syntactic analysis.

The NLP literature in the late 2010s has extensively explored probes to verify wheter embeddings and models shows linguistic properties. Meaning, if syntactic or semantic information can easily be extracted from the representations.

If you are curious about this literature, the following survey summarizes it:

[Yonatan Belinkov - Computational Linguistics 2022 - Probing Classifiers: Promises, Shortcomings,
and Advances](https://aclanthology.org/2022.cl-1.7/)

## 0. Setup

### 0.1 Setting up the environment

We use `uv` because it is faster, but you can use `pip`.

In [ ]:
# create the env
# uv venv

# activate the env
# source .venv/bin/activate  # linux
# .venv/Scripts/activate  # windows

# install required packages
# !uv pip install datasets interpreto

In [ ]:
# verify our installation
from bisect import bisect_right

import datasets
import torch
import torch.nn.functional as F
from matplotlib import colormaps
from matplotlib.colors import to_hex

import interpreto
from interpreto import plot_concepts


In [ ]:
if not torch.cuda.is_available():
    print("Warning: CUDA is not available. This may become quite long, we recommend tuning down the parameters.")
    device = "cpu"
else:
    device = "cuda"

### 0.2 Loading the dataset

We use the [Ontonotes dataset](https://huggingface.co/datasets/ontonotes/conll2012_ontonotesv5) from HuggingFace, in particular, its POS annotations. (Do not fear, the code does not download the full dataset).

Paper: [Pradhan et al. - CoNLL 2013 - Towards Robust Linguistic Analysis using {O}nto{N}otes](https://aclanthology.org/W13-3516/)

In [ ]:
from itertools import chain

nb_train = 10_000

dataset = datasets.load_dataset(
    'ontonotes/conll2012_ontonotesv5',
    data_dir='english_v12',
    revision='refs/convert/parquet',
    streaming=True,
)
tags = dataset['train'].features['sentences'].feature['pos_tags'].feature.names

data = list(chain.from_iterable(d["sentences"] for d in dataset["train"].take(2 * nb_train)))


We limit the number of classes to 14 to make things more interpretable to non-linguistic experts.

In [ ]:
COARSE_POS = {
    # Nouns
    "NN":   "NOUN",
    "NNS":  "NOUN",
    "NNP":  "NOUN",
    "NNPS": "NOUN",

    # Verbs and modals
    "VB":   "VERB",
    "VBD":  "VERB",
    "VBG":  "VERB",
    "VBN":  "VERB",
    "VBP":  "VERB",
    "VBZ":  "VERB",
    "VERB": "VERB",
    "MD":   "VERB",

    # Adjectives
    "JJ":  "ADJ",
    "JJR": "ADJ",
    "JJS": "ADJ",

    # Adverbs
    "RB":  "ADV",
    "RBR": "ADV",
    "RBS": "ADV",
    "WRB": "ADV",

    # Pronouns
    "PRP":  "PRON",
    "PRP$": "PRON",
    "WP":   "PRON",
    "WP$":  "PRON",
    "EX":   "PRON",

    # Determiners
    "DT":  "DET",
    "PDT": "DET",
    "WDT": "DET",

    # Adpositions
    "IN": "ADP",
    "TO": "ADP",

    # Conjunctions
    "CC": "CONJ",

    # Punctuation
    "``":    "PUNCT",
    "''":    "PUNCT",
    ",":     "PUNCT",
    ".":     "PUNCT",
    ":":     "PUNCT",
    "-LRB-": "PUNCT",
    "-RRB-": "PUNCT",
    "HYPH":  "PUNCT",
    "NFP":   "PUNCT",

    # Dataset/PTB-specific residual categories
    "XX":  "OTHER",
    "*":   "OTHER",
    "ADD": "OTHER",
    "AFX": "OTHER",
    "FW":  "OTHER",
    "LS":  "OTHER",
    "UH":  "OTHER",
    "$":   "OTHER",
    "SYM": "OTHER",
    "RP":  "OTHER",
    "POS": "OTHER",
    "CD":  "OTHER",
}

coarse_tags = sorted(set(COARSE_POS.values()))
# ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'OTHER','PRON', 'PUNCT', 'VERB']

assert set(tags) == set(COARSE_POS)

coarse_tag_to_id = {tag: i for i, tag in enumerate(coarse_tags)}

fine_to_coarse = torch.tensor([
    coarse_tag_to_id[COARSE_POS[tag]]
    for tag in tags
])

### 0.3 Loading the model

Please load and split a generation model using [`SplitterForGeneration`]() from interpreto.

- You can choose from HuggingFace generation models. But choose wisely depending on your hardware. You can use `"gpt2"` if you have no idea.
- Select the layer where to split your model. The earlier the lower the computations.
- Do not forget to set the `device`.
- Everything can be done in one line with interpreto.

In [ ]:
splitter = interpreto.SplitterForGeneration('Qwen/Qwen3-0.6B', 4, batch_size=16, device_map=device)

## 1. Probing POS

### 1.1 Alignment between tokenization and POS labels

One of the pain point of language explainability is tokenization. We face it directly here.

POS are labeled on words, and the model tokenizer does not consider each word as a token (even though, it is often the case in english with recent tokenizers).

In our case, if a word is split in several tokens, each token is assigned the word POS. If there are cases where a token compose several words, we discard the sample (this should be quite rare).

In addition, we need to group the POS words into correctly spaced sentences.

In [ ]:
CONTRACTIONS = {"n't", "'s", "'re", "'ve", "'ll", "'d", "'m"}

NO_SPACE_BEFORE = {
    ".", ",", "!", "?", ";", ":", "%",
    ")", "]", "}",
    *CONTRACTIONS,
}

NO_SPACE_AFTER = {
    "(", "[", "{",
    "$", "£", "€",
}

PTB_SURFACE = {
    "``": '"',
    "''": '"',
    "-LRB-": "(",
    "-RRB-": ")",
    "-LSB-": "[",
    "-RSB-": "]",
    "-LCB-": "{",
    "-RCB-": "}",
}


def detokenize(words):
    """Return natural-looking text and the character span of each PTB token."""
    text = ""
    spans = []

    for i, word in enumerate(words):
        surface = PTB_SURFACE.get(word, word)

        closing_quote = word == "''"

        if i == 0:
            sep = ""
        elif word in NO_SPACE_BEFORE or closing_quote:
            sep = ""
        elif words[i - 1] in NO_SPACE_AFTER or words[i - 1] == "``":
            sep = ""
        else:
            sep = " "

        text += sep
        start = len(text)
        text += surface
        spans.append((start, len(text)))

    return text, spans

In [ ]:
def align(sentences, tokenizer):
    texts, labels = [], []

    for sentence in sentences:
        text, spans = detokenize(sentence["words"])

        encoded = tokenizer(
            text,
            return_offsets_mapping=True,
            return_special_tokens_mask=True,
        )

        if len(encoded.input_ids) > tokenizer.model_max_length:
            continue

        sentence_labels = []

        for (start, end), special in zip(
            encoded.offset_mapping,
            encoded.special_tokens_mask,
            strict=True,
        ):
            if special:
                continue

            word = bisect_right([e for _, e in spans], start)

            # Model token overlaps several OntoNotes tokens.
            if word >= len(spans) or end > spans[word][1]:
                break

            sentence_labels.append(sentence["pos_tags"][word])

        else:
            texts.append(text)
            labels.append(sentence_labels)

    return texts, labels

In [ ]:
texts, labels = align(data, splitter.tokenizer)
labels = [fine_to_coarse[l] for l in labels]

count = 0
for text, label, sentence in zip(texts, labels, data, strict=True):
    print("Ontonotes elements:\t", sentence["words"])
    print("Ontonotes POS:\t\t", [coarse_tags[fine_to_coarse[l]] for l in sentence["pos_tags"]])
    print("Text:\t\t\t", text)
    print("Tokenized text:\t\t", splitter.tokenizer.tokenize(text))
    print("Tokenized POS:\t\t", [coarse_tags[l] for l in label])
    print()
    count += 1
    if count >= 3:
        break

### 1.2 Computing activations

Probes are trained on latent representations, also called activations.

Please rely on the [`SplitterForGeneration.get_activations()`](https://for-sight-ai.github.io/interpreto/api/concepts/splitters/splitter_for_generation/#interpreto.SplitterForGeneration.get_activations) method to compute activations for your training and test sets.

At the same time, we convert the POS labels to one hot encoding to learn each POS as a one versus all concept.

In [ ]:
# compute activations
x, _ = splitter.get_activations(texts, tqdm_bar=True)
x = x.to(device)

# flatten labels and apply one-hot encoding
y = torch.tensor([pos for sample_pos in labels for pos in sample_pos]).to(device)
y = F.one_hot(y, len(coarse_tags)).float()

assert x.shape[0] == y.shape[0], f"Mismatch between number of activations ({x.shape[0]}) and number of labels ({y.shape[0]})"

# split into train and test
x_train, y_train = x[:nb_train], y[:nb_train]
x_test, y_test = x[nb_train:], y[nb_train:]

### 1.3 Training a linear probe

Note that each POS class is learn independently of others. The probe here is $(l, d) \rightarrow (l, c)$, where $l$ is the number of tokens (can come from several samples), $d$ is the dimensionality of the activations, and $c$ is the number of POS classes.

In [ ]:
from interpreto.concepts.probes import LinearSVMProbe

probe = LinearSVMProbe(
    init_from_means_diff=False,
    lr=1e-3,
    max_iter=500,
)

probe.fit(x_train, y_train)

### 1.4 Evaluating the probe

In [ ]:
scores = probe.encode(x_test)

support = y_test.sum(0) / y_test.shape[0]
accuracies = torch.eq(scores > 0.0, y_test).float().mean(0)
for i, pos in enumerate(coarse_tags):
    print(f"{pos}: support: {support[i].item():.3f}, accuracy: {accuracies[i].item():.2f}")

### 1.5 Visualization

In [ ]:
test_ids = [0, 1, 2]
cmap = {i: to_hex(colormaps["tab10"](i)) for i in range(10)}

for i in test_ids:
    # list the text tokens
    text = texts[nb_train + i]
    tokens = [t.strip("Ġ") for t in splitter.tokenizer.tokenize(text)]

    # compute concept scores
    acts, _ = splitter.get_activations([text])
    scores = F.relu(probe.encode(acts.to(device)))

    # fancy plot
    plot_concepts(
        sample=tokens,
        concepts_activations=scores,
        concepts_importances=scores,
        concepts_labels=coarse_tags,
        default_colormap=cmap,
    )